# UR5 Evaluation Charts

Consistent styling for all evaluation visualizations (paper-ready).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import numpy as np

# ============================================================================
# SHARED DESIGN BLOCK
# Sets the look-and-feel for every chart in this notebook unless a chart
# cell explicitly overrides a value. Edit one place, all charts change.
# ============================================================================

# -- Paper-ready rcParams (font, sizes, axis colors) --
PI_STYLE = {
    'font.family': 'monospace',
    'font.monospace': ['DejaVu Sans Mono', 'Courier New'],
    'font.size': 18,
    'axes.labelsize': 23,
    'axes.titlesize': 22,
    'xtick.labelsize': 19,
    'ytick.labelsize': 19,
    'legend.fontsize': 17,
    'hatch.linewidth': 1.3,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'axes.linewidth': 1.0,
    'axes.edgecolor': '#444444',
    'xtick.major.width': 0.9,
    'ytick.major.width': 0.9,
    'xtick.major.size': 5,
    'ytick.major.size': 5,
    'xtick.color': '#333333',
    'ytick.color': '#333333',
    'axes.labelcolor': '#222222',
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
}
plt.rcParams.update(PI_STYLE)

# -- Neutral / chrome palette (text, gridlines, legend frame) --
TEXT_PRIMARY   = '#222222'  # axis labels, "Steps:", panel tags
TEXT_SECONDARY = '#5a5a5a'  # smaller tick text
TEXT_BAND      = '#9a9a9a'  # stage band labels (Approach/Grasp/Lift/Place)
TEXT_VALUE     = '#333333'  # value labels above bars
GRID_COLOR     = '#8a8578'
LEGEND_EDGE    = '#cfcfcf'

# -- Score chart palette (ID/OOD bars + loss line + overfit shade) --
COLOR_BAR_ID   = '#6b5a7f'  # darker plum (ID)
COLOR_BAR_OOD  = '#cabddc'  # lighter plum (OOD)
COLOR_BAR_EDGE = '#3d304f'
COLOR_LOSS      = '#fc8d62'
COLOR_LOSS_EDGE = '#d96d43'
COLOR_OVERFIT   = '#fde4d3'

# -- Stage palette (timing chart + any per-stage breakdown) --
STAGE_NAMES = ['Reach', 'Grasp', 'Transport', 'Release']
STAGE_COLORS = {
    'Reach':     ('#b3b3b3', '#6f6f6f'),  # gray
    'Grasp':     ('#fc8d62', '#a85532'),  # salmon
    'Transport': ('#8da0cb', '#4a5f8c'),  # periwinkle
    'Release':   ('#66c2a5', '#2f826a'),  # teal
}

# -- Ablation palette (dataset-size, horizon) --
# Named entries so each chart can map its own categories to a meaning.
ABLATION_COLORS = {
    'gray':       ('#b3b3b3', '#6f6f6f'),
    'salmon':     ('#fc8d62', '#a85532'),
    'teal':       ('#66c2a5', '#2f826a'),  # hero
    'periwinkle': ('#8da0cb', '#4a5f8c'),
    'pink':       ('#e78ac3', '#8a3e76'),
    'gold':       ('#ffd92f', '#9c8300'),
}

# -- Sizing / line weights --
STAGE_LABEL_SIZE = 16
STEP_LABEL_SIZE  = 18
BAR_VALUE_SIZE   = 16
PANEL_TAG_SIZE   = 18
GRID_LW          = 0.7
GRID_ALPHA       = 0.75
BAR_LW           = 0.8

# -- Standard legend frame (omit columnspacing so each chart can set its own) --
LEGEND_KWARGS = dict(
    frameon=True, facecolor='white', edgecolor=LEGEND_EDGE,
    framealpha=0.94, fancybox=True, borderpad=0.6,
    handlelength=1.6, handletextpad=0.6,
)

SAVE_DIR = '.'  # charts saved alongside notebook


def paper_axes(ax, *, x_ticks=False, y_ticks=False, gridlines=True):
    """Apply the shared paper-style frame to `ax`:
       - hide top/left/right spines
       - drop tick marks (length=0); tick LABELS stay
       - draw horizontal gridlines if `gridlines=True`
    Pass x_ticks=True / y_ticks=True to keep tick marks on that axis.
    """
    for s in ('top', 'left', 'right'):
        ax.spines[s].set_visible(False)
    ax.set_axisbelow(True)
    if not x_ticks:
        ax.tick_params(axis='x', length=0)
    if not y_ticks:
        ax.tick_params(axis='y', length=0)
    if gridlines:
        ax.grid(axis='y', which='major', color=GRID_COLOR,
                linewidth=GRID_LW, alpha=GRID_ALPHA, zorder=0)


def add_stage_bands(ax, names=None, ys=(0.22, 0.47, 0.72, 0.97),
                    *, x=0.005, color=None, italic=True):
    """Inner-left stage band labels at the y-fractions in `ys`.
    Defaults to STAGE_NAMES and TEXT_BAND.
    """
    if names is None:
        names = STAGE_NAMES
    if color is None:
        color = TEXT_BAND
    trans = ax.get_yaxis_transform()
    for name, y in zip(names, ys):
        ax.text(x, y, name, ha='left', va='top',
                transform=trans, fontsize=STAGE_LABEL_SIZE, color=color,
                fontstyle='italic' if italic else 'normal', zorder=0)


def add_step_labels(ax, steps, centers, *, prefix=None,
                    prefix_x=-0.13, y=-0.03, title=None, title_y=-0.10):
    """Below-axis "120 150 180 ..." row used by the score and timing charts.
    `centers` are the data x-coordinates of each group. `title`, if given,
    is drawn centered below the row (acts like an x-axis label).
    """
    trans = ax.get_xaxis_transform()
    for step, cx in zip(steps, centers):
        ax.text(cx, y, f'{step}', ha='center', va='top',
                transform=trans, fontsize=STEP_LABEL_SIZE, color=TEXT_PRIMARY)
    if prefix:
        ax.text(prefix_x, y, prefix, ha='right', va='top',
                transform=trans, fontsize=STEP_LABEL_SIZE, color=TEXT_PRIMARY)
    if title:
        ax.text(0.5, title_y, title, ha='center', va='top',
                transform=ax.transAxes,
                fontsize=plt.rcParams['axes.labelsize'],
                color=TEXT_PRIMARY)


def fmt_mmss(total_s):
    mm, ss = divmod(int(round(total_s)), 60)
    return f'{mm}:{ss:02d}'


## ID vs OOD scores + training loss

In [ ]:
# -- Real eval data from pi05_ur5_blueblock10-1 (see ur5/docs/final_evaluations.md) --
# Scores: mean task-success score across 5 trials per checkpoint per condition.
# ID = block inside the ~30 cm training area. OOD = block outside it.
# Step 90 omitted — policy hadn't learned to approach the object yet.
steps = [120, 150, 180, 210]

id_scores  = [0.00, 0.80, 0.65, 0.75]
ood_scores = [0.00, 0.70, 0.65, 0.25]
loss       = [0.0105, 0.0085, 0.0071, 0.0084]

# Per-checkpoint min/max across the 5 trials (for error bars).
id_min  = [0.00, 0.50, 0.25, 0.75]
id_max  = [0.00, 1.00, 0.75, 0.75]
ood_min = [0.00, 0.50, 0.25, 0.25]
ood_max = [0.00, 0.75, 0.75, 0.25]

In [ ]:
# ID vs OOD task progress + training loss across checkpoints.
fig, ax = plt.subplots(figsize=(9.0, 7.4))

x_scale = 0.50
x = np.arange(len(steps)) * x_scale
width = 0.13
gap = 0.04

ZERO_STUB = 0.012
id_arr  = np.array(id_scores,  dtype=float)
ood_arr = np.array(ood_scores, dtype=float)
id_heights  = np.where(id_arr  > 0, id_arr,  ZERO_STUB)
ood_heights = np.where(ood_arr > 0, ood_arr, ZERO_STUB)
id_colors  = [COLOR_BAR_ID  if v > 0 else COLOR_BAR_EDGE for v in id_arr]
ood_colors = [COLOR_BAR_OOD if v > 0 else COLOR_BAR_EDGE for v in ood_arr]

ax.bar(x - width/2 - gap/2, id_heights, width,
       color=id_colors, alpha=0.95,
       edgecolor=COLOR_BAR_EDGE, linewidth=BAR_LW)
ax.bar(x + width/2 + gap/2, ood_heights, width,
       color=ood_colors, alpha=0.95,
       edgecolor='#3a3a3a', linewidth=BAR_LW, hatch='////')

id_lo = id_arr - np.array(id_min)
id_hi = np.array(id_max) - id_arr
ood_lo = ood_arr - np.array(ood_min)
ood_hi = np.array(ood_max) - ood_arr
ax.errorbar(x - width/2 - gap/2, id_arr, yerr=[id_lo, id_hi],
            fmt='none', ecolor='#2a2235', capsize=4, lw=1.2, zorder=5)
ax.errorbar(x + width/2 + gap/2, ood_arr, yerr=[ood_lo, ood_hi],
            fmt='none', ecolor='#2a2235', capsize=4, lw=1.2, zorder=5)

ax.set_xlabel('')
ax.set_ylabel('Average task progress', labelpad=12)
ax.set_xticks([])
add_step_labels(ax, steps, x, title='Training steps')

ax.set_xlim(-x_scale * 0.55, x[-1] + x_scale * 0.55)
ax.set_ylim(0, 1.10)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.50))
ax.yaxis.set_minor_locator(mticker.MultipleLocator(0.25))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

add_stage_bands(ax)

# Training loss on a twin axis.
ax_twin = ax.twinx()
ax_twin.plot(x, loss, color=COLOR_LOSS, linestyle='-', marker='o',
             linewidth=1.4, markersize=7.5, markeredgecolor=COLOR_LOSS_EDGE,
             markeredgewidth=1.4, label='Training loss', alpha=0.95, zorder=5)
ax_twin.set_ylabel('Training loss', labelpad=12)
ax_twin.set_ylim(0, 0.0132)
ax_twin.yaxis.set_major_locator(mticker.MultipleLocator(0.006))
ax_twin.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))

# Overfitting region (right side of axis, around the last checkpoint).
overfit_start = (x[-2] + x[-1]) / 2
overfit_end   = x[-1] + x_scale * 0.5
ax.axvspan(overfit_start, overfit_end, alpha=0.25, color=COLOR_OVERFIT,
           edgecolor='none', zorder=0)
ax.axvspan(overfit_start, overfit_end, facecolor='none', hatch='////',
           edgecolor=COLOR_LOSS_EDGE, alpha=0.10, linewidth=0, zorder=0.5)
ax.axvline(overfit_start, color=COLOR_LOSS_EDGE, linewidth=1.1,
           linestyle='-', alpha=0.55, zorder=1)
ax.text(0.935, 0.70, 'Overfitting', ha='center', va='center',
        transform=ax.transAxes, rotation=90, fontsize=17,
        color=COLOR_LOSS_EDGE, zorder=6)

# Legend below axis.
id_patch  = mpatches.Patch(facecolor=COLOR_BAR_ID,  edgecolor=COLOR_BAR_EDGE,
                           linewidth=BAR_LW, label='In-distribution')
ood_patch = mpatches.Patch(facecolor=COLOR_BAR_OOD, edgecolor='#3a3a3a',
                           linewidth=BAR_LW, hatch='////', label='Out-of-distribution')
loss_handles, loss_labels = ax_twin.get_legend_handles_labels()
ax.legend([id_patch, ood_patch] + loss_handles,
          ['In-distribution', 'Out-of-distribution'] + loss_labels,
          loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=3, columnspacing=2.0, **LEGEND_KWARGS)

paper_axes(ax)
ax.grid(axis='y', which='minor', color=GRID_COLOR, linewidth=GRID_LW,
        alpha=GRID_ALPHA, zorder=0)
ax.tick_params(axis='y', which='minor', length=0)
paper_axes(ax_twin, gridlines=False)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eval_id_vs_ood.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/eval_id_vs_ood.pdf', bbox_inches='tight')
plt.show()


## Dataset size ablation — task progress

One bar per number of fine-tuning episodes, colored by condition. Placeholder
values below — replace with real eval results.

In [ ]:
# -- Dataset size ablation data (placeholders) --
# Rows: one per number of fine-tuning episodes.
# Columns: the four 0.25-weighted progress stages.
dataset_sizes = [0, 1, 10, 20, 30, 50]

# Stage credit per column (each stage ≤ 0.25 so the full task sums to 1.0)
# 10 ep = step 150 ID from final_evaluations.md (5/5, 5/5, 4/5, 2/5).
stage_approach = [0.00, 0.00, 0.25, 0.25, 0.25, 0.25]
stage_grasp    = [0.00, 0.00, 0.25, 0.00, 0.00, 0.00]
stage_lift     = [0.00, 0.00, 0.20, 0.00, 0.00, 0.00]
stage_place    = [0.00, 0.00, 0.10, 0.00, 0.00, 0.00]

# Per-bar min/max across trials (only 10 ep has variance).
ds_min = [0.00, 0.00, 0.50, 0.25, 0.25, 0.25]
ds_max = [0.00, 0.00, 1.00, 0.25, 0.25, 0.25]

In [ ]:
# Dataset size ablation - one bar per # episodes, no stacking.
DATASET_PALETTE = {
    0:  ABLATION_COLORS['gray'],
    1:  ABLATION_COLORS['salmon'],
    10: ABLATION_COLORS['teal'],        # hero
    20: ABLATION_COLORS['periwinkle'],
    30: ABLATION_COLORS['pink'],
    50: ABLATION_COLORS['gold'],
}

fig, ax = plt.subplots(figsize=(7.4, 5.4))

x_scale = 0.50
x = np.arange(len(dataset_sizes)) * x_scale
width = 0.32

totals = (np.array(stage_approach) + np.array(stage_grasp)
          + np.array(stage_lift) + np.array(stage_place))

ZERO_STUB = 0.018
for i, size in enumerate(dataset_sizes):
    fill, edge = DATASET_PALETTE[size]
    if totals[i] == 0:
        ax.bar(x[i], ZERO_STUB, width, color=edge, alpha=0.75,
               edgecolor=edge, linewidth=BAR_LW, zorder=2)
    else:
        ax.bar(x[i], totals[i], width, color=fill, alpha=0.95,
               edgecolor=edge, linewidth=0.9, zorder=2)

add_stage_bands(ax)

ax.set_xlabel('Dataset size (episodes)', labelpad=10)
ax.set_ylabel('Average task progress', labelpad=12)
ax.set_xticks(x)
ax.set_xticklabels(dataset_sizes)
ax.set_xlim(-x_scale * 0.55, x[-1] + x_scale * 0.55)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.25))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

ds_lo = totals - np.array(ds_min)
ds_hi = np.array(ds_max) - totals
ax.errorbar(x, totals, yerr=[ds_lo, ds_hi], fmt='none',
            ecolor='#2a2235', capsize=4, lw=1.2, zorder=5)

def _darken(hex_, f=0.5):
    h = hex_.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'#{int(r*f):02x}{int(g*f):02x}{int(b*f):02x}'
for xi, size, total, lo, hi in zip(x, dataset_sizes, totals, ds_min, ds_max):
    has_err = hi - lo > 0
    tx = xi + (width * 0.08 if has_err else 0)
    ha = 'left' if has_err else 'center'
    if total > 0 or has_err:
        ax.text(tx, total + 0.005, f'{int(round(total * 100))}%',
                ha=ha, va='bottom',
                fontsize=BAR_VALUE_SIZE, color='#3a3a3a')

paper_axes(ax)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eval_dataset_size.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/eval_dataset_size.pdf', bbox_inches='tight')
plt.show()


## Action horizon ablation — task progress

Task success as a function of the action-chunk horizon executed at inference
(how many predicted actions are played before the next policy call).
Placeholder values below — replace with real eval results.

In [ ]:
# -- Action horizon ablation: real data from final_evaluations.md (H sweep, ID, ckpt 150) --
# Rows: one per horizon length (# predicted actions executed per inference call).
# Columns: the four 0.25-weighted progress stages.
horizon_steps = [3, 6, 9, 12, 15]

# Per-stage success fraction × 0.25 (each stage ≤ 0.25 so the full task sums to 1.0)
h_approach = [0.25, 0.25, 0.25, 0.25, 0.25]  # 5/5 reach across all horizons
h_grasp    = [0.00, 0.25, 0.10, 0.00, 0.00]  # 0/5, 5/5, 2/5, 0/5, 0/5
h_lift     = [0.00, 0.20, 0.10, 0.00, 0.00]  # 0/5, 4/5, 2/5, 0/5, 0/5
h_place    = [0.00, 0.10, 0.05, 0.00, 0.00]  # 0/5, 2/5, 1/5, 0/5, 0/5

# Per-horizon min/max across the 5 trials (for error bars).
h_min = [0.25, 0.50, 0.25, 0.25, 0.25]
h_max = [0.25, 1.00, 1.00, 0.25, 0.25]

In [ ]:
# Action horizon ablation - one bar per HORIZON_STEPS value.
HORIZON_PALETTE = {
    3:  ABLATION_COLORS['gray'],        # too short
    6:  ABLATION_COLORS['teal'],        # hero - best at 0.90
    9:  ABLATION_COLORS['periwinkle'],
    12: ABLATION_COLORS['gold'],
    15: ABLATION_COLORS['salmon'],      # too long
}

fig, ax = plt.subplots(figsize=(8.5, 5.4))

x_scale = 0.65
x = np.arange(len(horizon_steps)) * x_scale
width = 0.32

h_totals = (np.array(h_approach) + np.array(h_grasp)
            + np.array(h_lift) + np.array(h_place))

ZERO_STUB = 0.018
for i, h in enumerate(horizon_steps):
    fill, edge = HORIZON_PALETTE[h]
    if h_totals[i] == 0:
        ax.bar(x[i], ZERO_STUB, width, color=edge, alpha=0.75,
               edgecolor=edge, linewidth=BAR_LW, zorder=2)
    else:
        ax.bar(x[i], h_totals[i], width, color=fill, alpha=0.95,
               edgecolor=edge, linewidth=0.9, zorder=2)

add_stage_bands(ax)

ax.set_xlabel('Actions executed per inference call', labelpad=10)
ax.set_ylabel('Average task progress', labelpad=12)
ax.set_xticks(x)
ax.set_xticklabels(horizon_steps)

ax.set_xlim(-x_scale * 0.95, x[-1] + x_scale * 0.55)
ax.set_ylim(0, 1.08)
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.25))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1.0, decimals=0))

def _darken(hex_, f=0.5):
    h = hex_.lstrip('#')
    r, g, b = int(h[0:2],16), int(h[2:4],16), int(h[4:6],16)
    return f'#{int(r*f):02x}{int(g*f):02x}{int(b*f):02x}'
h_lo = h_totals - np.array(h_min)
h_hi = np.array(h_max) - h_totals
ax.errorbar(x, h_totals, yerr=[h_lo, h_hi], fmt='none',
            ecolor='#2a2235', capsize=4, lw=1.2, zorder=5)

for xi, h, total, lo, hi in zip(x, horizon_steps, h_totals, h_min, h_max):
    has_err = hi - lo > 0
    tx = xi + (width * 0.08 if has_err else 0)
    ha = 'left' if has_err else 'center'
    if total > 0 or has_err:
        ax.text(tx, total + 0.005, f'{int(round(total * 100))}%',
                ha=ha, va='bottom',
                fontsize=BAR_VALUE_SIZE, color='#3a3a3a')

paper_axes(ax)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eval_horizon.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/eval_horizon.pdf', bbox_inches='tight')
plt.show()


## Per-stage trial duration across checkpoints

Mean seconds spent in each stage (Reach / Grasp / Transport / Release), averaged
over the trials that reached the stage. Shows the policy not only improves
in success rate but also becomes markedly faster — and the step-210 OOD
collapse is visible as a very short "Approach only" bar.

In [ ]:
# Per-stage trial timings from pi05_ur5_blueblock10-1 evaluations.
# Source: ur5/docs/final_evaluations.md summary table.
# Values are MEAN SECONDS spent in each stage, averaged over trials that
# reached that stage. 0 = no trial reached that stage at that checkpoint.
time_steps = [150, 180, 210]

t_id  = np.array([
    [57, 157, 60, 128],  # step 150 ID  (n reaching: 5, 4, 4, 2)
    [36,  41, 58,  0],   # step 180 ID  (n reaching: 5, 4, 4, 0)
    [38,  26, 65,  0],   # step 210 ID  (n reaching: 5, 5, 5, 0)
], dtype=float)
t_ood = np.array([
    [55, 108, 93, 0],    # step 150 OOD (n reaching: 5, 5, 4, 0)
    [41,  78, 77, 0],    # step 180 OOD (n reaching: 5, 4, 4, 0)
    [52,   0,  0, 0],    # step 210 OOD (n reaching: 5, 0, 0, 0)
], dtype=float)

fig, ax = plt.subplots(figsize=(9.0, 7.4))

bar_w = 0.13
pair_gap = 0.04
group_gap = 0.41

positions_id, positions_ood, group_centers = [], [], []
for i in range(len(time_steps)):
    base = i * (2 * bar_w + pair_gap + group_gap)
    positions_id.append(base)
    positions_ood.append(base + bar_w + pair_gap)
    group_centers.append(base + (bar_w + pair_gap) / 2)
positions_id    = np.array(positions_id)
positions_ood   = np.array(positions_ood)
group_centers   = np.array(group_centers)

# Stacked bars by stage. ID = solid fill; OOD = same colors with hatch.
for s_idx, stage in enumerate(STAGE_NAMES):
    fill, edge = STAGE_COLORS[stage]
    bot_id  = t_id[:, :s_idx].sum(axis=1)  if s_idx > 0 else np.zeros(len(time_steps))
    bot_ood = t_ood[:, :s_idx].sum(axis=1) if s_idx > 0 else np.zeros(len(time_steps))
    ax.bar(positions_id, t_id[:, s_idx], bar_w, bottom=bot_id,
           color=fill, edgecolor=edge, linewidth=BAR_LW,
           label=stage, zorder=2)
    ax.bar(positions_ood, t_ood[:, s_idx], bar_w, bottom=bot_ood,
           color=fill, edgecolor='#3a3a3a', linewidth=BAR_LW,
           hatch='////', zorder=2)

# Total trial duration label above each bar.
for pos, total in zip(positions_id, t_id.sum(axis=1)):
    if total > 0:
        ax.text(pos, total + 5, fmt_mmss(total), ha='center', va='bottom',
                fontsize=BAR_VALUE_SIZE, color=TEXT_VALUE)
for pos, total in zip(positions_ood, t_ood.sum(axis=1)):
    if total > 0:
        ax.text(pos, total + 5, fmt_mmss(total), ha='center', va='bottom',
                fontsize=BAR_VALUE_SIZE, color=TEXT_VALUE)

ax.set_xticks([])
add_step_labels(ax, time_steps, group_centers, title='Training steps')

ax.set_xlabel('')
ax.set_ylabel('Trial duration (mm:ss)', labelpad=12)
ax.set_ylim(0, 420)
ax.yaxis.set_major_locator(mticker.MultipleLocator(180))
ax.yaxis.set_major_formatter(mticker.FuncFormatter(
    lambda s, pos: f'{int(s // 60)}:{int(s % 60):02d}'))
ax.set_xlim(-bar_w - 0.10, positions_ood[-1] + bar_w + 0.10)

# Legend: stage colors + condition (ID/OOD) below the axis, single row.
stage_handles = [
    mpatches.Patch(facecolor=STAGE_COLORS[s][0],
                   edgecolor=STAGE_COLORS[s][1], label=s)
    for s in STAGE_NAMES
]
cond_handles = [
    mpatches.Patch(facecolor='white', edgecolor='#333333', linewidth=1.4,
                   label='In-distribution'),
    mpatches.Patch(facecolor='white', edgecolor='#3a3a3a', linewidth=1.4,
                   hatch='////', label='Out-of-distribution'),
]
ax.legend(handles=stage_handles + cond_handles,
          loc='upper center', bbox_to_anchor=(0.5, -0.20),
          ncol=3, columnspacing=1.6, **LEGEND_KWARGS)

paper_axes(ax)

plt.tight_layout()
plt.savefig(f'{SAVE_DIR}/eval_timing.png', dpi=300, bbox_inches='tight')
plt.savefig(f'{SAVE_DIR}/eval_timing.pdf', bbox_inches='tight')
plt.show()
